In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np

import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

import seaborn as sns

from sklearn.cluster import KMeans
import torch

from sklearn.decomposition import PCA
import networkx as nx
from torch_geometric.utils import to_networkx

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader

from torch_geometric.explain.config import ModelConfig
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch2_data():
    """
    Loads (192,3) data from 'Feature_CNN2_Feature_CNN2_target_struct_without_transcript.txt'.
    """
    data_branch2 = []
    current_array = []
    with open('Feature_CNN2_Feature_CNN2_target_struct_without_transcript.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 192, 3)
    return X_branch2

def load_reaction_rates():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)
    
########################################
# 2. Graph Data Utilities (for GNN)
########################################

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

########################################
# 2. CNN1 Branch
########################################

class CNNBranch2(nn.Module):
    """
    CNN branch for (192,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*96, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,96)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,96)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


########################################
# 3. Final Fusion Model
########################################

class CNN2_Only(nn.Module):
    """
    End-to-end: 
      - CNNBranch2 => feat_cnn2
    dropout => final FC => 1
    """
    def __init__(self,
                 filters2, kernel_size2, dense_units2,  # CNN2
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        
        # total dimension = (dense_units2)
        total_dim = dense_units2
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)


    def forward(self, x2):
        feat_cnn2 = self.cnn_branch2(x2)                   # (batch, dense_units2)

        feat_cnn2 = F.dropout(feat_cnn2, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(feat_cnn2))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)

########################################
# 4. Hybrid Dataset
########################################

class IndivDataset(Dataset):
    def __init__(self, X2, reaction_rates):
        super().__init__()
        self.X2 = X2
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(X2) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        x2 = torch.tensor(self.X2[idx], dtype=torch.float) 
        y = torch.tensor(self.reaction_rates[idx], dtype=torch.float)
        return x2, y

def collate(batch):
    from torch_geometric.data import Batch
    x2_list, y_list = zip(*batch)
    x2 = torch.stack(x2_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x2, y


class FullModelWrapperForCNN2(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.full_model = full_model.eval()

    def forward(self, x2):
        batch_size = x2.shape[0]

        out = self.full_model(x2)
        return out.view(-1, 1)  # Return shape: (batch, 1)


def filter_extreme_by_column(matrix, threshold=0.2):
    """
    For each column in the matrix:
    - If the range (max - min) exceeds the threshold,
    - Keep only the max and min values.
    - Set other values to 0.
    """
    filtered = np.zeros_like(matrix)
    for col in range(matrix.shape[1]):
        col_vals = matrix[:, col]
        col_max = np.max(col_vals)
        col_min = np.min(col_vals)
        if (col_max - col_min) >= threshold:
            max_idx = np.argmax(col_vals)
            min_idx = np.argmin(col_vals)
            filtered[max_idx, col] = col_max
            filtered[min_idx, col] = col_min
    return filtered


import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import seaborn as sns
import numpy as np
import pandas as pd


# Segment ranges: (start, end)
segments_cnn2 = [(0, 66), (66, 96), (96, 162), (162, 192)]

np.random.seed(42)
torch.manual_seed(42)


# === Example Execution ===
if __name__ == "__main__":
    # Load your trained model
    full_model = torch.load(
    "CNN2_without_transcript_trial_14.pt",
    weights_only=False
)
    full_model.eval()

    from sklearn.cluster import KMeans
    from sklearn.metrics import pairwise_distances_argmin_min
    import torch
    import numpy as np
    
    # Load original data
    X2_all = load_branch2_data()
    rates = load_reaction_rates()
    
    # Flatten for clustering
    X2_flat = X2_all.reshape(X2_all.shape[0], -1)
    
    # Run KMeans
    kmeans2_sample = KMeans(n_clusters=1000, random_state=42).fit(X2_flat)
    kmeans2_background = KMeans(n_clusters=100, random_state=42).fit(X2_flat)
    
    # Get indices of closest original points to each centroid
    closest_X2_indices_sample, _ = pairwise_distances_argmin_min(kmeans2_sample.cluster_centers_, X2_flat)
    closest_X2_indices_background, _ = pairwise_distances_argmin_min(kmeans2_background.cluster_centers_, X2_flat)
    
    # Get samples 
    X2_selected = X2_all[closest_X2_indices_sample]

    # Get backgrounds
    background_cnn2 = X2_all[closest_X2_indices_background]
    
    # Use these indices to extract the corresponding reaction rates
    rates_X2 = rates[closest_X2_indices_sample]

    # Fixed samples in different branches (sample is the one with median reaction rate)
    sample_idx = np.argsort(rates)[len(rates) // 2]  # index of median value
    x2_sample = torch.tensor(X2_all[sample_idx], dtype=torch.float).unsqueeze(0)  

    # === CNN Branch 2 SHAP ===
    
    # 1. Create the wrapper
    wrapper = FullModelWrapperForCNN2(
        full_model=full_model)
    
    # 2. Build background and input tensors
    background_cnn2 = torch.tensor(background_cnn2, dtype=torch.float) 
    X2_tensor = torch.tensor(X2_selected, dtype=torch.float)    
    X2_all_tensor = torch.tensor(X2_all, dtype=torch.float)   
    
    # 3. Run SHAP
    explainer = shap.GradientExplainer(wrapper, background_cnn2)
    shap_values_cnn2 = explainer.shap_values(X2_tensor)
    print("SHAP shape:", shap_values_cnn2[0].shape)

    wrapper.eval()
    
    shap_array = shap_values_cnn2
    shap_array = shap_array.squeeze(-1)  
    avg_shap = shap_array.mean(axis=0) 

    heatmap_data = avg_shap.T 
    global_vmin = heatmap_data.min()
    global_vmax = heatmap_data.max()

    filtered_heatmap_data = filter_extreme_by_column(heatmap_data, threshold=global_vmax*0.1)

    positions = np.arange(1, 193)
    states = ["Unpaired", "Paired-Opening", "Paired-Closing"]
    feature_names_cnn2 = [f"Pos{pos-66}_{state}" for pos in positions for state in states]
    
    X2_flat = X2_tensor.view(X2_tensor.size(0), -1).numpy()
    shap_values_cnn2_flat = shap_values_cnn2.squeeze().reshape(X2_tensor.size(0), -1)
    
    segments_cnn2_ssDNA_target_bh = [(66, 96)]
    for seg_start, seg_end in segments_cnn2_ssDNA_target_bh:
        seg_feature_indices = list(range(seg_start * 3, seg_end * 3))  # 3 features per position
        seg_feature_names = [feature_names_cnn2[i] for i in seg_feature_indices]
        
        import pandas as pd
        X_values = X2_flat[:, seg_feature_indices].cpu().numpy() if hasattr(X2_flat[:, seg_feature_indices], "cpu") else X2_flat[:, seg_feature_indices]
        
        df_input = pd.DataFrame(X_values, columns=seg_feature_names)
        df_shap = pd.DataFrame( shap_values_cnn2_flat[:, seg_feature_indices], columns=seg_feature_names)

        df_combined = pd.concat([df_input, df_shap], axis=1)

        df_combined.to_csv("indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_14.csv", index=False)

print('finished')

In [ ]:
# Models tested: 14, 20, 44, 46, 51, 52, 61, 66, 82, 89

In [ ]:
import pandas as pd
import csv
from scipy.stats import spearmanr
from collections import Counter

# --- Config ---
file_names = [
    "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_14.csv", "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_20.csv",
    "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_44.csv", "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_46.csv",
    "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_51.csv", "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_52.csv",
    "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_61.csv", "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_66.csv",
    "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_82.csv", "indiv_shap_values_cnn2_ssDNA_target_bh_RfxCas13a_validation_target_struct_without_transcript_89.csv"
]

results = []

# Step 1: collect results
for file_name in file_names:
    model_id = file_name.split("_")[-1].replace(".csv", "")  # Extract model ID like '27'

    with open(file_name) as f:
        reader = csv.reader(f)
        header = next(reader)

    df = pd.read_csv(file_name, header=None, skiprows=1)
    df.columns = header  # overwrite with true header

    col_counts = Counter(df.columns)
    duplicate_features = [col for col, count in col_counts.items() if count > 1]

    for feature in duplicate_features:
        indices = [i for i, col in enumerate(df.columns) if col == feature]
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                col1 = df.iloc[:, indices[i]]
                col2 = df.iloc[:, indices[j]]
                corr, pval = spearmanr(col1, col2)
                results.append({
                    "Feature": feature,
                    "Model": model_id,
                    "Spearman Correlation": corr,
                    "P-Value": pval
                })

# Step 2: Build DataFrame and order features
df = pd.DataFrame(results)

# Track first appearance order of features
feature_order = []
seen = set()
for row in results:
    feat = row["Feature"]
    if feat not in seen:
        seen.add(feat)
        feature_order.append(feat)

# Use category sorting to preserve order
df["Feature_order"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)
df = df.sort_values(by=["Feature_order", "Model"])
df = df.drop(columns=["Feature_order"])

# Step 3: Blank out repeated feature names
def clear_repeated_features(df):
    cleared = []
    current_feat = None
    for _, row in df.iterrows():
        feat = row["Feature"]
        if feat == current_feat:
            row["Feature"] = ""
        else:
            current_feat = feat
        cleared.append(row)
    return pd.DataFrame(cleared)

df_clean = clear_repeated_features(df)

# Step 4: Save CSV
df_clean.to_csv("indiv_spearman_duplicates_grouped_norepeat_RfxCas13a_validation_target_struct_without_transcript_cnn2_ssDNA_target_bh.csv", index=False)
print("Saved with feature names grouped and de-duplicated to 'indiv_spearman_duplicates_grouped_norepeat_RfxCas13a_validation_target_struct_without_transcript_cnn2_ssDNA_target_bh.csv'")
